# Reporting: plots, scores and the submission file

The code that looks at results lives here; the run notebooks and `helper_predict.ipynb` call it.

* `evaluate_on_gt(labels, probs, gt, clean, ...)` scores labels against a ground truth of any length, given as a `(T,)`
  array with `X` where there is no label. It reports the score over all labelled frames and, when the file is longer
  than `chunk_s`, per chunk of `chunk_s` seconds (`CHUNK_S`, 300 by default; 0 turns chunks off).
* Plots go to `outputs/figures/`. `plot_gt_report` draws timelines (ground truth, decoded labels and per-frame argmax
  as colour strips, with errors, ball tracking and probabilities) and a page of confusion matrices, spellF1 against
  IoU, per-possession matches, calibration and accuracy by ball-tracking state. `plot_decoder_report` covers the
  held-out external matches, the decoder grid and possession lengths.
* `write_submission` writes the `frame,label` file, reads it back and checks it the way `validate_submission.py` does.
  `possession_stats` shows whether the sequence flickers.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_decoder.ipynb", globals())
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

POSS_COLOR = {"H": "#2f6fdb", "A": "#e8743b", "D": "#b9bdc6", "X": "#ffffff"}
POSS_NAME = {"H": "Home", "A": "Away", "D": "Dead ball", "X": "Abstain (X)"}
ERR_COLOR = {"R": "#cfe8cf", "W": "#c0392b", "X": "#ffffff"}
BALL_COLOR = {"O": "#333333", "M": "#e3e3e3"}
FPS = 25


def fill_halves(lbl: np.ndarray, blocks) -> np.ndarray:
    """Labels after the scorer's forward-fill of X (per half) - what spellF1 actually sees."""
    out = lbl.copy()
    for _, a, b in blocks:
        out[a:b + 1] = ffill_labels(lbl[a:b + 1])
    return out


def _draw_runs(ax, lbl: np.ndarray, a: int, b: int, colors: dict, t0: float = 0.0) -> None:
    """Coloured bar per run of identical labels over frames [a, b]; x axis in minutes, frame a sitting at ``t0``."""
    spans = {k: [] for k in colors}
    for l, s, e in label_runs(lbl[a:b + 1]):
        spans[l].append((t0 + s / FPS / 60, (e - s) / FPS / 60))
    for k, sp in spans.items():
        if sp:
            ax.broken_barh(sp, (0, 1), facecolors=colors[k], edgecolor="none")
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)


def plot_timeline(rows: dict, gt_key: str, a: int, b: int, blocks, probs=None, ball_src=None, title: str = "", path=None) -> None:
    """One coloured strip per label sequence (ground truth on top), an error strip per prediction (red = wrong),
    whether the ball was actually seen, and the classifier's probabilities underneath. Frames [a, b]."""
    t0 = a / FPS / 60                                                       # x axis = minutes of tracking time, so each window shows where it sits in the match
    names = list(rows)
    preds = [n for n in names if n != gt_key]
    gt = rows[gt_key]
    filled = {n: fill_halves(rows[n], blocks) for n in preds}
    ratios = [1.0] * len(names) + [0.45] * len(preds) + ([0.45] if ball_src is not None else []) + ([2.4] if probs is not None else [])
    fig, axes = plt.subplots(len(ratios), 1, sharex=True, figsize=(14, 1.9 + 0.45 * len(names) + 0.25 * len(preds) + (2.2 if probs is not None else 0)),
                             gridspec_kw={"height_ratios": ratios}, layout="constrained")
    axes = np.atleast_1d(axes)
    k = 0
    for n in names:
        _draw_runs(axes[k], rows[n], a, b, POSS_COLOR, t0)
        axes[k].set_ylabel(n, rotation=0, ha="right", va="center")
        k += 1
    for n in preds:
        g, p = gt[a:b + 1], filled[n][a:b + 1]
        err = np.where(g == "X", "X", np.where(p == g, "R", "W"))
        _draw_runs(axes[k], np.concatenate([err, ["X"]]), 0, len(err) - 1, ERR_COLOR, t0)
        wrong = 100 * (err == "W").sum() / max((err != "X").sum(), 1)
        axes[k].set_ylabel(f"errors | {n}\n({wrong:.0f}% of frames wrong)", rotation=0, ha="right", va="center", fontsize=8)
        k += 1
    if ball_src is not None:
        _draw_runs(axes[k], np.concatenate([np.where(ball_src[a:b + 1] == 0, "O", "M"), ["M"]]), 0, b - a, BALL_COLOR, t0)
        axes[k].set_ylabel("ball tracked?", rotation=0, ha="right", va="center", fontsize=8)
        k += 1
    if probs is not None:
        P = np.nan_to_num(probs[a:b + 1])
        step = max(1, len(P) // 4000)
        t = t0 + np.arange(len(P))[::step] / FPS / 60
        axes[k].stackplot(t, P[::step, 0], P[::step, 1], P[::step, 2], colors=[POSS_COLOR[c] for c in "HAD"], linewidth=0)
        axes[k].set_ylim(0, 1)
        axes[k].set_ylabel("classifier\nP(H) P(A) P(D)", rotation=0, ha="right", va="center", fontsize=8)
    axes[-1].set_xlim(t0, t0 + (b - a + 1) / FPS / 60)
    axes[-1].set_xlabel("minutes of tracking time")
    handles = [Patch(facecolor=POSS_COLOR[c], edgecolor="#999999", label=POSS_NAME[c]) for c in "HADX"]
    handles += [Patch(facecolor=ERR_COLOR["R"], label="correct"), Patch(facecolor=ERR_COLOR["W"], label="wrong")]
    if ball_src is not None:
        handles += [Patch(facecolor=BALL_COLOR["O"], label="ball seen"), Patch(facecolor=BALL_COLOR["M"], edgecolor="#999999", label="ball reconstructed")]
    axes[0].legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.04), ncol=len(handles), frameon=False)
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def confusion(gt: np.ndarray, pred: np.ndarray, blocks) -> np.ndarray:
    """(3, 3) frame counts, rows = ground truth H/A/D, columns = prediction H/A/D (X forward-filled first)."""
    p, m = fill_halves(pred, blocks), gt != "X"
    return np.array([[int(((gt == g) & (p == q) & m).sum()) for q in "HAD"] for g in "HAD"])


def spell_table(pred: np.ndarray, gt: np.ndarray, blocks):
    """For every ground-truth possession: the best temporal IoU with a predicted possession of the same team. -> (DataFrame, n predicted)."""
    out, n_pred = [], 0
    for _, a, b in blocks:
        g = gt[a:b + 1]
        sc = g != "X"
        if not sc.any():
            continue
        where = np.flatnonzero(sc)
        P = [x for x in label_runs(ffill_labels(pred[a:b + 1])[sc]) if x[0] in ("H", "A")]
        n_pred += len(P)
        for l, s, e in label_runs(g[sc]):
            if l not in ("H", "A"):
                continue
            best = 0.0
            for pl, ps, pe in P:
                if pl == l:
                    inter = max(0, min(e, pe) - max(s, ps))
                    best = max(best, inter / ((e - s) + (pe - ps) - inter))
            out.append({"team": l, "start": a + int(where[s]), "dur_s": (e - s) / FPS, "best_iou": best})
    return pd.DataFrame(out), n_pred


def possession_durations(lbl: np.ndarray, blocks) -> np.ndarray:
    return np.array([(e - s) / FPS for _, a, b in blocks for l, s, e in label_runs(ffill_labels(lbl[a:b + 1])) if l in ("H", "A")])


def _heat_confusion(ax, cm: np.ndarray, title: str) -> None:
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:,}\n{100 * pct[i, j]:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if pct[i, j] > 0.55 else "black")
    ax.set_xticks(range(3), [POSS_NAME[c] for c in "HAD"])
    ax.set_yticks(range(3), [POSS_NAME[c] for c in "HAD"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("ground truth")
    ax.set_title(title, loc="left", fontsize=9, fontweight="bold")
    for s in ax.spines.values():
        s.set_visible(False)


def plot_eval_dashboard(gt, decoded, argmax_lbl, probs, blocks, ball_src, title: str = "", path=None) -> None:
    """2 x 3 panels: confusion matrices, spellF1 vs IoU threshold, per-possession IoU, calibration, accuracy by ball status."""
    fig, ax = plt.subplots(2, 3, figsize=(16, 9), layout="constrained")
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    variants = {"Decoded (HMM)": (decoded, "#1b7f5c"), "Per-frame argmax": (argmax_lbl, "#8a8a8a")}

    _heat_confusion(ax[0, 0], confusion(gt, decoded, blocks), "Confusion | decoded (rows sum to 100%)")
    _heat_confusion(ax[0, 1], confusion(gt, argmax_lbl, blocks), "Confusion | per-frame argmax")

    ths = np.round(np.arange(0.1, 1.0, 0.1), 2)
    a_ = ax[0, 2]
    a_.axvspan(0.3, 0.7, color="#f1f1f1", zorder=0)
    for name, (lbl, c) in variants.items():
        a_.plot(ths, [spell_f1(lbl, gt, blocks, thresholds=(t,)) for t in ths], "o-", color=c, label=name)
    a_.set_xlabel("IoU threshold for a possession to count as matched")
    a_.set_ylabel("possession F1")
    a_.set_ylim(0, 1.02)
    a_.set_title("spellF1 vs IoU threshold (shaded = thresholds the metric averages)", loc="left", fontsize=9, fontweight="bold")
    a_.legend(frameon=False)

    tab, n_pred = spell_table(decoded, gt, blocks)
    a_ = ax[1, 0]
    if len(tab):
        a_.bar(range(len(tab)), tab.best_iou, color=[POSS_COLOR[t] for t in tab.team], edgecolor="#555555", linewidth=0.5)
        for th in (0.3, 0.5, 0.7):
            a_.axhline(th, color="#555555", lw=0.8, ls="--")
            a_.text(1.01, th, f"IoU {th}", va="center", fontsize=7, transform=a_.get_yaxis_transform())
        t0 = blocks[0][1]
        step = max(1, len(tab) // 10)                                       # long ground truth: label every few bars only
        a_.set_xticks(list(range(0, len(tab), step)), [f"{int((s - t0) / FPS // 60)}:{int((s - t0) / FPS % 60):02d}\n{d:.0f}s"
                                                      for s, d in zip(tab.start.iloc[::step], tab.dur_s.iloc[::step])], fontsize=7)
    a_.set_ylim(0, 1.05)
    a_.set_ylabel("best IoU with a predicted possession")
    a_.set_xlabel("ground-truth possession (start time | length)")
    a_.set_title(f"Per-possession match | decoded ({len(tab)} true vs {n_pred} predicted)", loc="left", fontsize=9, fontweight="bold")

    # reliability of the classifier probabilities on the labelled frames
    a_ = ax[1, 1]
    code = np.array([{"H": 0, "A": 1, "D": 2}.get(g, -1) for g in gt])
    m = (code >= 0) & np.isfinite(probs).all(1)
    conf, ok = probs[m].max(1), probs[m].argmax(1) == code[m]
    edges = np.linspace(1 / 3, 1, 9)
    bi = np.clip(np.digitize(conf, edges) - 1, 0, 7)
    cnt = np.bincount(bi, minlength=8)
    acc = np.array([ok[bi == i].mean() if cnt[i] else np.nan for i in range(8)])
    mid = np.array([conf[bi == i].mean() if cnt[i] else np.nan for i in range(8)])
    ece = float(np.nansum(cnt / cnt.sum() * np.abs(acc - mid)))
    a_.plot([0, 1], [0, 1], color="#999999", ls="--", lw=1)
    a_.bar(mid, 0.18 * cnt / cnt.max(), width=0.05, color="#999999", alpha=0.35)
    a_.plot(mid, acc, "o-", color="#1b7f5c")
    a_.set_xlim(0.3, 1.0)
    a_.set_ylim(0, 1.02)
    a_.set_xlabel("classifier confidence (max probability)")
    a_.set_ylabel("share of frames that were right")
    a_.set_title(f"Calibration | ECE {ece:.3f} (grey bars = how many frames)", loc="left", fontsize=9, fontweight="bold")

    # accuracy by how well the ball was tracked
    a_ = ax[1, 2]
    cats = [("all frames", np.ones(len(gt), bool)), ("ball seen", ball_src == 0),
            ("short gap\n(interpolated)", np.isin(ball_src, (1, 2))), ("long gap\n(player-cluster guess)", np.isin(ball_src, (3, 4)))]
    w = 0.38
    for i, (name, (lbl, c)) in enumerate(variants.items()):
        f = fill_halves(lbl, blocks)
        vals = [(f[(gt != "X") & mk] == gt[(gt != "X") & mk]).mean() if ((gt != "X") & mk).any() else np.nan for _, mk in cats]
        bars = a_.bar(np.arange(len(cats)) + (i - 0.5) * w, vals, w, color=c, label=name)
        a_.bar_label(bars, labels=[f"{v:.2f}" for v in vals], fontsize=7)
    a_.set_xticks(range(len(cats)), [f"{n}\nn={int(((gt != 'X') & mk).sum()):,}" for n, mk in cats], fontsize=8)
    a_.set_ylim(0, 1.25)
    a_.set_ylabel("frame accuracy")
    a_.set_title("Accuracy by ball tracking quality", loc="left", fontsize=9, fontweight="bold")
    a_.legend(frameon=False, loc="upper center", ncol=2)
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_decoder_report(val_ids, probs_by_match, labels_by_match, blocks_by_match, cfg, tuning=None, pub=None, path=None):
    """Held-out external matches (+ optionally BAR-ATH's ground truth as `pub = (gt, decoded, probs, gt_blocks)`):
    per-match scores, decoder tuning grid, possession-length distributions. Returns {match: decoded labels}."""
    decoded, scores = {}, {}
    for m in val_ids:
        decoded[m] = HMMSequenceDecoder(cfg).decode_match(probs_by_match[m], blocks_by_match[m]).labels
        scores[m] = (score_labels(argmax_labels(probs_by_match[m], blocks_by_match[m]), labels_by_match[m], blocks_by_match[m]),
                     score_labels(decoded[m], labels_by_match[m], blocks_by_match[m]))
    if pub is not None:
        g, d, p, bl = pub
        scores["BAR-ATH\n(ground truth)"] = (score_labels(argmax_labels(p, bl), g, bl), score_labels(d, g, bl))
    fig, ax = plt.subplots(2, 2, figsize=(13, 8.5), layout="constrained")
    fig.suptitle("Decoder and generalisation", x=0.01, ha="left", fontweight="bold")
    x = np.arange(len(scores))
    for a_, key in zip(ax[0], ("spellF1", "macroF1")):
        for i, (nm, c) in enumerate((("Per-frame argmax", "#8a8a8a"), ("Decoded (HMM)", "#1b7f5c"))):
            v = [s[i][key] for s in scores.values()]
            bars = a_.bar(x + (i - 0.5) * 0.38, v, 0.38, color=c, label=nm)
            a_.bar_label(bars, labels=[f"{t:.2f}" for t in v], fontsize=7)
        a_.set_xticks(x, list(scores), fontsize=8)
        a_.set_ylim(0, 1.2)
        a_.set_ylabel(key)
        a_.set_title(f"{key} per match", loc="left", fontsize=9, fontweight="bold")
    ax[0, 0].legend(frameon=False, loc="upper center", ncol=2, fontsize=8)

    a_ = ax[1, 0]
    if tuning is not None:
        piv = tuning.pivot(index="min_possession_s", columns="emission_scale", values="spellF1").sort_index()
        im = a_.imshow(piv.values, cmap="viridis", aspect="auto", origin="lower")
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                a_.text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if piv.values[i, j] < np.nanmax(piv.values) * 0.85 else "black")
        a_.set_xticks(range(piv.shape[1]), [f"{c:g}" for c in piv.columns])
        a_.set_yticks(range(piv.shape[0]), [f"{r:g}" for r in piv.index])
        a_.set_xlabel("emission_scale (higher = trust the classifier more)")
        a_.set_ylabel("min possession (s)")
        fig.colorbar(im, ax=a_, label="spellF1 (external validation)", shrink=0.8)
        a_.set_title("Decoder tuning grid", loc="left", fontsize=9, fontweight="bold")
    else:
        a_.axis("off")

    a_ = ax[1, 1]
    series = {"truth | external val": (np.concatenate([possession_durations(labels_by_match[m], blocks_by_match[m]) for m in val_ids]), "#222222", "-"),
              "decoded | external val": (np.concatenate([possession_durations(decoded[m], blocks_by_match[m]) for m in val_ids]), "#1b7f5c", "-")}
    if pub is not None:
        series["truth | BAR-ATH"] = (possession_durations(pub[0], pub[3]), "#222222", ":")
        series["decoded | BAR-ATH"] = (possession_durations(pub[1], pub[3]), "#1b7f5c", ":")
    for nm, (d, c, ls) in series.items():
        d = np.sort(d)
        a_.step(d, np.arange(1, len(d) + 1) / len(d), where="post", color=c, ls=ls, label=f"{nm} (n={len(d)})")
    a_.set_xscale("log")
    a_.set_xlabel("possession length (s)")
    a_.set_ylabel("share of possessions shorter than x")
    a_.set_title("Possession length distribution (flicker shows up as too many short ones)", loc="left", fontsize=9, fontweight="bold")
    a_.legend(frameon=False, fontsize=8)
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return decoded


def plot_gt_report(tag: str, name: str, decoded: np.ndarray, probs: np.ndarray, gt: np.ndarray, clean, chunk_s: Optional[float] = None,
                   max_windows: Optional[int] = None) -> None:
    """Timeline(s) + dashboard against a ground truth of any length. The timelines show the first ``max_windows`` windows of ``chunk_s`` seconds (None = all of them)
    (10 minutes when chunk_s is 0; all of the ground truth when there is less than that); the dashboard covers all of it. ``tag`` goes into the file names."""
    chunk_s = CHUNK_S if chunk_s is None else chunk_s
    blocks = gt_blocks(gt, clean.half_blocks)
    arg = argmax_labels(probs, clean.half_blocks)
    sc = score_labels(decoded, gt, blocks)
    head = (f"{name} | spellF1 {sc['spellF1']:.3f} | macroF1 {sc['macroF1']:.3f} | "
            f"{sc['poss_pred']} possessions predicted vs {sc['poss_gt']} true")
    rows = {"Ground truth": gt, "Decoded (HMM)": decoded, "Per-frame argmax": arg}
    windows = chunk_blocks(blocks, chunk_s)[:max_windows] if chunk_s else []
    if not windows:
        cap = int((chunk_s or 600) * FPS)
        windows = [(h, a, min(b, a + cap - 1)) for h, a, b in blocks[:max_windows]]
    for k, (_, a, b) in enumerate(windows, 1):
        plot_timeline(rows, "Ground truth", a, b, blocks, probs=probs, ball_src=clean.ball_src, title=head,
                      path=FIG_DIR / (f"{tag}_timeline.png" if len(windows) == 1 else f"{tag}_timeline_{k}.png"))
    plot_eval_dashboard(gt, decoded, arg, probs, blocks, clean.ball_src, title=head, path=FIG_DIR / f"{tag}_dashboard.png")

In [ ]:
CHUNK_S = 300.0        # default length of a scoring chunk in seconds (run_decoder.ipynb / helper_predict.ipynb pass their own; 0 = score the ground truth as one piece)


def gt_blocks(gt: np.ndarray, half_blocks=None) -> List[Tuple[int, int, int]]:
    """(half, first, last) of every stretch the ground truth covers (not X); cut at the half boundaries when ``half_blocks`` is given."""
    m = (np.asarray(gt) != "X").astype(np.int8)
    edges = np.diff(np.concatenate([[0], m, [0]]))
    out = []
    for s, e in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1):
        if half_blocks is None:
            out.append((len(out) + 1, int(s), int(e)))
            continue
        for h, a, b in half_blocks:
            lo, hi = max(int(s), a), min(int(e), b)
            if lo <= hi:
                out.append((h, lo, hi))
    return out


def chunk_blocks(blocks, chunk_s: float) -> List[Tuple[int, int, int]]:
    """Every ground-truth stretch cut into pieces of ``chunk_s`` seconds (what is left over at the end is not scored)."""
    n = int(round(chunk_s * FPS))
    return [(h, a + k * n, a + (k + 1) * n - 1) for h, a, b in blocks for k in range((b - a + 1) // n)]


def evaluate_on_gt(labels: np.ndarray, probs: np.ndarray, gt: np.ndarray, clean, tag: str = "eval", name: str = "model",
                   chunk_s: Optional[float] = None, plots: bool = True, max_windows: Optional[int] = None) -> dict:
    """Scores decoded ``labels`` (and the per-frame argmax of ``probs``) against a ground truth of ANY length (a ``(T,)`` array, X = not covered):
    over everything it covers and, when it is longer than ``chunk_s``, per chunk of ``chunk_s`` seconds (the labels are decoded over whole halves
    first, as in the real scoring). ``chunk_s`` None = CHUNK_S, 0 = no chunks. ``plots`` draws the timelines + dashboard (see plot_gt_report)."""
    chunk_s = CHUNK_S if chunk_s is None else chunk_s
    gt = np.asarray(gt)
    blocks = gt_blocks(gt, clean.half_blocks)
    covered = int((gt != "X").sum())
    print(f"ground truth covers {covered:,} frames = {covered / FPS / 60:.1f} min in {len(blocks)} stretch(es)")
    y = np.full(len(gt), -1, np.int8)
    for lab_, code_ in LABEL_TO_ID.items():
        y[gt == lab_] = code_
    on = np.flatnonzero(y >= 0)
    print("per frame:", frame_scores(probs[on], y[on]))
    arg = argmax_labels(probs, clean.half_blocks)
    table = pd.DataFrame([{"variant": f"{name} | per-frame argmax", **score_labels(arg, gt, blocks)},
                          {"variant": f"{name} | decoded", **score_labels(labels, gt, blocks)}])
    print(table.to_string(index=False))
    out = {"table": table, "chunks": None}
    chunks = chunk_blocks(blocks, chunk_s) if chunk_s else []
    if len(chunks) > 1:
        rows = []
        for h, a, b in chunks:
            d = score_labels(labels, gt, [(h, a, b)])
            rows.append({"half": h, "start_min": round(a / FPS / 60, 1), **{k: d[k] for k in ("spellF1", "macroF1", "poss_pred", "poss_gt")}})
        ch = pd.DataFrame(rows)
        se = ch.spellF1.std(ddof=1) / np.sqrt(len(ch))
        print(f"\nper {chunk_s:g}-s chunk ({len(ch)} chunks): spellF1 {ch.spellF1.mean():.4f} +/- {se:.4f} | macroF1 {ch.macroF1.mean():.4f}")
        if len(ch) <= 40:
            print(ch.to_string(index=False))
        out["chunks"] = ch
    if plots:
        plot_gt_report(tag, name, labels, probs, gt, clean, chunk_s, max_windows)
    return out

In [ ]:
SUBMISSION_COLUMNS = ["frame", "label"]


def validate_submission_frame(df: pd.DataFrame, n_frames: int) -> List[str]:
    """Same contract as SUBMISSION.md / validate_submission.py. Returns a list of problems (empty = fine)."""
    problems = []
    if list(df.columns) != SUBMISSION_COLUMNS:
        problems.append(f"header must be exactly {SUBMISSION_COLUMNS}, got {list(df.columns)}")
        return problems
    if len(df) != n_frames:
        problems.append(f"expected {n_frames} rows, got {len(df)}")
    if not np.array_equal(df["frame"].to_numpy(), np.arange(len(df))):
        problems.append("`frame` must be 0..N-1 in order")
    bad = sorted(set(df["label"].astype(str)) - {"H", "A", "D", "X"})
    if bad:
        problems.append(f"labels outside H/A/D/X: {bad}")
    if df["label"].isna().any():
        problems.append("blank labels (write X to abstain)")
    return problems


def write_submission(labels: np.ndarray, path: Path) -> pd.DataFrame:
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({"frame": np.arange(len(labels)), "label": labels})
    df.to_csv(path, index=False)
    back = pd.read_csv(path, keep_default_na=False)          # 'N/A'-style strings must not become NaN
    problems = validate_submission_frame(back, len(labels))
    if problems:
        raise ValueError("submission failed validation: " + "; ".join(problems))
    return back


def possession_stats(labels: np.ndarray, half_blocks, fps: int = 25) -> Dict[str, float]:
    """Counts after the scorer's forward-fill. A mean duration far below ~14 s means the sequence is flickering."""
    n, durs, counts = 0, [], {k: 0 for k in "HADX"}
    for _, a, b in half_blocks:
        seg = labels[a:b + 1]
        for k in counts:
            counts[k] += int((seg == k).sum())
        poss = [(e - s) / fps for l, s, e in label_runs(ffill_labels(seg)) if l in ("H", "A")]
        n, durs = n + len(poss), durs + poss
    graded = sum(counts.values())
    return {"possessions": n, "mean_possession_s": round(float(np.mean(durs)), 1) if durs else 0.0,
            **{f"frac_{k}": round(counts[k] / max(graded, 1), 3) for k in "HADX"}}

In [ ]:
def plot_prediction_summary(gt: np.ndarray, pred: np.ndarray, blocks, probs=None, ball_src=None,
                            chunk_s: float = 300.0, max_windows: Optional[int] = None, title: str = "", path=None) -> None:
    """Exactly two views:
      (1) CONFUSION MATRIX - ground-truth rows vs predicted columns (H/A/D), X forward-filled first.
      (2) PER-FRAME TIMELINE - coloured strips comparing, for every frame, what the model PREDICTS against the
          ground truth (blue = Home, orange = Away, grey = Dead), with a red 'errors' strip where the prediction is
          wrong. The match is drawn in windows of ``chunk_s`` seconds so the whole thing is legible.
    ``tag`` for the files comes from ``path`` (the confusion matrix gets a '_confusion' suffix, the timelines '_timeline[_k]')."""
    base = Path(path) if path is not None else None

    # (1) confusion matrix
    fig, ax = plt.subplots(figsize=(4.6, 4.2), layout="constrained")
    if title:
        fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    _heat_confusion(ax, confusion(gt, pred, blocks), "Confusion matrix (rows sum to 100%)")
    if base is not None:
        fig.savefig(base.with_name(base.stem + "_confusion" + base.suffix), dpi=150, bbox_inches="tight")
    plt.show()

    # (2) per-frame timeline: ground truth vs prediction, over the whole match in chunk_s windows
    rows = {"Ground truth": gt, "Predicted": pred}
    windows = chunk_blocks(blocks, chunk_s) if chunk_s else blocks
    windows = windows[:max_windows] if max_windows else windows
    for k, (_, a, b) in enumerate(windows, 1):
        p_ = base.with_name(base.stem + f"_timeline_{k}" + base.suffix) if base is not None else None
        plot_timeline(rows, "Ground truth", a, b, blocks, probs=probs, ball_src=ball_src,
                      title=(title + f" - frames {a}-{b}") if title else f"frames {a}-{b}", path=p_)
